In [1]:
import pandas as pd
import numpy as np

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data -O data-week-3.csv

--2026-09-29 20:09:18--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-09-29 20:09:18 (140 MB/s) - ‘data-week-3.csv’ saved [278746/278746]



In [ ]:
df = pd.read_csv('data-week-3.csv')
df.head(1).T

,0
lead_source,organic_search
industry,technology
employment_status,employed
location,europe
annual_income,88160.0
number_of_courses_viewed,3
interaction_count,4
lead_score,0.64
converted,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [ ]:
#check for empty values
df.isna().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [ ]:
#check the unique values for some columns. This is just an example
df.lead_score.unique()

array([0.64, 0.72, 0.58, 0.57, 0.62, 0.77, 0.78, 0.41, 0.52, 0.35, 0.11,
       0.33, 0.13, 0.85, 0.55, 0.56, 0.5 , 0.4 , 0.47, 0.73, 0.76, 0.74,
       0.53, 0.6 , 0.32, 0.34, 0.83, 0.37, 0.48, 0.42, 0.38, 0.3 , 0.49,
       0.54, 0.16, 0.84, 0.63, 0.28, 0.46, 0.67, 0.44, 0.51, 0.71, 0.81,
       0.66, 0.39, 0.75, 0.22, 0.01, 0.21, 0.31, 0.45, 0.23, 0.43, 0.68,
       0.59, 0.18, 0.17, 0.19, 0.27, 0.26, 0.65, 0.99, 0.69, 0.86, 0.07,
       0.89, 0.24, 0.36, 0.05, 0.92, 0.14, 0.06, 0.97, 0.7 , 0.93, 0.87,
       0.88, 0.15, 0.09, 0.61, 0.29, 0.79,  nan, 0.2 , 0.04, 0.8 , 0.03,
       0.98, 0.25, 0.91, 0.82, 0.96, 0.94, 0.1 , 0.12, 0.9 , 0.08, 0.02,
       0.95])

In [26]:
#keep name of categorical columns for one-hot encoding and numbers seperately
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
numerical_columns = list(df.select_dtypes(include=['number']).columns)
numerical_columns.remove('converted')

In [ ]:
# Fills  categorical columns with 'NA' and numerical columns with 0
df[categorical_columns] = df[categorical_columns].fillna('N/A')
df[numerical_columns] = df[numerical_columns].fillna(0)

In [ ]:
#check the # of unique values for categorical columns. This is the exact number of possible new columns (if they got accepted as a feature) due to one-hot encoding
df[categorical_columns].nunique()

,0
lead_source,6
industry,7
employment_status,5
location,6


In [ ]:
#technology
df.industry.mode()

,industry
0,technology


In [ ]:
#lead_score
df[numerical_columns].corrwith(df.interaction_count).rename('Correlation').to_frame('Correlation').sort_values(by='Correlation', ascending=False)

,Correlation
interaction_count,1.000000
lead_score,0.915746
number_of_courses_viewed,0.721609
converted,0.448624
annual_income,0.122842


In [ ]:
#interaction_count
df[numerical_columns].corrwith(df.lead_score).rename('Correlation').to_frame('Correlation').sort_values(by='Correlation', ascending=False)

#so it lead_score and interaction_count

,Correlation
lead_score,1.000000
interaction_count,0.915746
number_of_courses_viewed,0.757204
converted,0.483246
annual_income,0.229496


In [ ]:
from sklearn.model_selection import train_test_split

#60% training, 20% validation, 20% test
#df_full_train has 80% of the dataset
#df_train has 60% of the dataset
#df_val has 20% of the dataset
#df_test has 20% of the dataset
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [ ]:
#due to shuffle of records, we reste the indexing
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

In [ ]:
#converter is the target variable that's why we keep it separately and remove this from our train/test/val datasets
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [ ]:
#most of the people got converted (57%)
print(df_full_train.converted.value_counts(normalize=True))
print(df_full_train.converted.mean()) #it's the same with the ones that got converted (always)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64
0.57075


In [ ]:
from sklearn.metrics import mutual_info_score

#lead_source
for i in categorical_columns:
  print(i, ': ', mutual_info_score(df_full_train.converted, df_full_train[i]))

lead_source :  0.0347609795100606
industry :  0.0028984818739526547
employment_status :  0.02031283511067701
location :  0.0010498214288891111


In [80]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)

def LogisticRegressionTrain(tmp_list, C):
  train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
  X_train = dv.fit_transform(train_dict)

  # The validation set must be encoded in exactly the same way as the training set.
  # That is why we call fit only on the training data - for the validation data we reuse the already fitted vectorizer and call only transform.
  # If we called fit again on validation, the columns could come out in a different order - or a category that appears only in validation would create a new column.
  # The matrix would no longer match the one the model was trained on.
  val_dict = df_val[tmp_list].to_dict(orient='records')
  X_val = dv.transform(val_dict)
  #train the model using specific parameters
  model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)

  #w0 = model.intercept_[0]
  #wn = model.coef_[0].round(3)
  y_pred = model.predict_proba(X_val)[:, 1]

  # Convert probabilities to binary predictions at the 0.5 threshold
  y_pred_binary = (y_pred >= 0.5).astype(int)
  return y_pred_binary, y_val

def CalculateAccuracy (y_pred_binary, y_val): # Calculate accuracy directly
  accuracy = (y_pred_binary == y_val).mean()
  return accuracy

In [81]:
y_pred_binary, y_val = LogisticRegressionTrain(categorical_columns + numerical_columns, C = 1.0)
global_accuracy = CalculateAccuracy(y_pred_binary, y_val)
print('Global Accuracy is: ', round(global_accuracy, 2))

Global Accuracy is:  0.64


In [82]:
tmp_list = []

for i in categorical_columns:
  tmp_list = categorical_columns.copy()
  tmp_list.remove(i)
  y_pred_binary, y_val = LogisticRegressionTrain(tmp_list + numerical_columns, C = 1.0)
  accuracy = CalculateAccuracy(y_pred_binary, y_val)
  print(i, ':', accuracy)
  print('Difference with global:', abs(global_accuracy - accuracy), '\n')

for i in numerical_columns:
  tmp_list = numerical_columns.copy()
  tmp_list.remove(i)
  y_pred_binary, y_val = LogisticRegressionTrain(tmp_list + categorical_columns, C = 1.0)
  accuracy = CalculateAccuracy(y_pred_binary, y_val)
  print(i, ':', accuracy)
  print('Difference with global:', abs(global_accuracy - accuracy), '\n')

  #lead_source has the smallest_difference

lead_source : 0.643
Difference with global: 0.0020000000000000018 

industry : 0.644
Difference with global: 0.0010000000000000009 

employment_status : 0.645
Difference with global: 0.0 

location : 0.644
Difference with global: 0.0010000000000000009 

annual_income : 0.624
Difference with global: 0.02100000000000002 

number_of_courses_viewed : 0.656
Difference with global: 0.01100000000000001 

interaction_count : 0.433
Difference with global: 0.21200000000000002 

lead_score : 0.647
Difference with global: 0.0020000000000000018 



In [83]:
# List of C values to test
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in C_values:
  y_pred_binary, y_val = LogisticRegressionTrain(categorical_columns + numerical_columns, C = c)
  print(c, ':', CalculateAccuracy(y_pred_binary, y_val))

1e-06 : 0.598
1e-05 : 0.598
0.0001 : 0.613
0.001 : 0.645
